# 5. 下單

**下單 (Order)** 指所有送出、刪除與修改委託的相關操作。

透過自身程式邏輯分析市場後，最終都要轉化成下單動作，並在市場上建立部位

API 下單分為兩種，以功能可區分為一般委託 (Order) 和智慧單 (Smart Order)

------

## 5.1. 一般委託

一般委託有三種操作：送單、刪單與改單。

### 5.1.1. 送出委託

要送出一般證券委託委託非常容易，透過 `[商品檔].order(...).send()` 即可

如果你要指定證券帳號，就在 `send()` 中放入使用者 ID 

在還沒有 send 之前, 你會拿到 `PendingOrder`, Send 出去後會拿到的是 `Order`


In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, User

api.logger.show = False

# 註冊事件
@api.event.order.send_start
def onOrderSendStart(data):
    print(f"Order send start: {data}")
    
@api.event.order.send_fail
def onOrderSendFail(data):
    print(f"Order send fail: {data}")
    
@api.event.order.send_success
def onOrderSendSuccess(data):
    print(f"Order send success: {data}")

@api.event.order.placed_success
def onOrderPlacedSuccess(data):
    print(f"Order placed success: {data}")

@api.event.order.placed_fail
def onOrderPlacedFail(data):
    print(f"Order placed fail: {data}")
     
@api.event.order.changed
def onOrderChanged(data):
    print("Order Changed", data)

@api.event.order.cancel_fail
def onCancelOrderFailed(data):
    print(f"Cancel Order Failed", data)


api.init()
user: User = api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2888"]
order = stock.order(api.const.ORDER.ACTION.BUY, 10, 1).send(user.id)

print(f"\n\norder: {order}")
print(f"order.price: {order.price}")
print(f"order.status: {order.status}")
print(f"order.qty: {order.qty}")
print(f"order.deal_qty: {order.deal_qty}")
print(f"order.remain_qty: {order.remain_qty}")
print(f"order.canceled_qty: {order.canceled_qty}")

api.keepalive() # 為了看到報價, 不讓 Notebook 停止

%ZP INFO  2024-08-20 09:36:10.411169 | Main | 26356 |[ eskmo version: 0.0.88 ]
Order send start: OrderSendStartResult(account='91829808465', symbol='2888', exchange=0, period=0, order_flag=0, buysell=0, price=10, qty=1, price_type=0, trade_type=2, callbackId=0)
Order send success: OrderSendSuccessResult(account='91829808465', symbol='2888', exchange=0, period=0, order_flag=0, buysell=0, price=10, qty=1, price_type=0, trade_type=2, state='Pending', created=datetime.datetime(2024, 8, 20, 9, 36, 31, 497000), callbackId=0, threadId='34764')
Order Changed OrderChangedResult(count=1, order=OrderStatus(reply=Reply(num=1, key_no='4308201055408', market='TS', type='委託', status='成功', broker='9182', cust_no='9808465', buysell_info='B00R2', exchange_id='TW', symbol='2888', strike_price='', book_no='T00QA', price='10.0000', numerator='', denominator='', price_lags=[ReplyPrice(price='', numerator='', denominator=''), ReplyPrice(price='', numerator='', denominator='')], volume='1000', before_qty='0',


### 5.1.2. 刪除委託

透過 `order.cancel()` 即可刪除該筆委託

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, User

api.logger.show = False

# 註冊事件
@api.event.order.changed
def onOrderChanged(data):
    print("Order Changed", data)

@api.event.order.cancel_fail
def onCancelOrderFailed(data):
    print(f"Cancel Order Failed", data)


api.init()
user: User = api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2888"]

order = stock.order(api.const.ORDER.ACTION.BUY, 10, 1).send(user.id)
print(order)

order.cancel()
print(order)

api.keepalive() # 為了看到報價, 不讓 Notebook 停止

%BP INFO  2024-08-20 10:12:20.968568 | Main | 24764 |[ eskmo version: 0.0.88 ]
Order Changed OrderChangedResult(count=3, order=OrderStatus(reply=Reply(num=3, key_no='4308201090589', market='TS', type='委託', status='成功', broker='9182', cust_no='9808465', buysell_info='B00R2', exchange_id='TW', symbol='2888', strike_price='', book_no='T00X3', price='10.0000', numerator='', denominator='', price_lags=[ReplyPrice(price='', numerator='', denominator=''), ReplyPrice(price='', numerator='', denominator='')], volume='1000', before_qty='0', after_qty='1000', date_str='20240820', time_str='10:12:43', ok_seq='', sub_id='0000000', sale_no='8890', agent='y', trade_date='20240820', msg_no='1010000586868', pre_order='A', commodity_lags=[ReplyCommodity(com_id='2888', year_month='', strike_price=''), ReplyCommodity(com_id='', year_month='', strike_price='')], execution_no='', price_symbol='', reserved='', order_effective='', call_put='', order_seq='', error_msg='', cancel_order_mark_by_exchange='', exch


### 5.1.3. 修改委託

透過 `order.modify(price=..., qty=...)` 即可修改該筆委託

> <br/>
> 建置中
> <br/><br/>

------

## 5.2. 智慧單委託

智慧單為券商提供特殊的下單服務，以便讓人更方便的實現交易邏輯

智慧單有很多種，Eskmo 支持的市場與單別包含：

| 市場 | 單別                                                                                                                          | 備註 |
|------|-------------------------------------------------------------------------------------------------------------------------------|------|
| 證券 | [MIT](#), [OCO](#), [CB](#) |      |
| 期貨 | MIT, OCO, CB                                                                                                                  |      |


### 5.2.1. 送出智慧單委託

智慧單送單與一般委託相似，只是 `.order` 換為智慧單的單別 `.smartOrder.mit`



In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, User, MITOrderSendStartResult, MITOrderSendFailResult, MITOrderSendSuccessResult

api.logger.show = False

# 註冊事件
# MIT Order
@api.event.mit_order.send_start
def onMITOrderSendStart(data: MITOrderSendStartResult):
    print(f"MIT order send start: '{data}'")

@api.event.mit_order.send_fail
def onMITOrderSendFail(data: MITOrderSendFailResult):
    print("MIT order send fail", data)

@api.event.mit_order.send_success
def onMITOrderSendSuccess(data: MITOrderSendSuccessResult):
    print(f"MIT order send success: {data}")

@api.event.mit_order.placed_fail
def onMITOrderPlacedFail(data):
    print("mit order placed fail", data)  

@api.event.mit_order.placed_success
def onMITOrderPlacedSuccess(data):
    print("mit order placed success", data)  

@api.event.mit_order.changed
def onMITOrderChanged(data):
    print("Smart Order Changed", data)

@api.event.mit_order.cancel_fail
def onMITCancelOrderFailed(data):
    print(f"Cancel MIT Order Failed", data)


api.init()
user: User = api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2888"]
smartOrder = stock.smartOrder.mit(api.const.ORDER.ACTION.BUY, 10, 1, 10).send(user.id) 
print(smartOrder) 

api.keepalive() # 不讓 Notebook 停止

%WP INFO  2024-08-20 10:22:35.611030 | Main | 15916 |[ eskmo version: 0.0.88 ]
MIT order send start: 'MITOrderSendStartResult(account='91829808465', symbol='2888', buysell=0, price=10, qty=1, trigger_price=10, trigger_dir=2, order_flag=0, price_type=2, trade_type=0, is_pre_trade_risk_controlled=False, is_gtc_order=False, gtc_date='', gtc_end_by=1, callbackId=0)'
MIT order send success: MITOrderSendSuccessResult(account='91829808465', symbol='2888', buysell=0, price=10, qty=1, trigger_price=10, trigger_dir=2, order_flag=0, price_type=2, trade_type=0, is_pre_trade_risk_controlled=False, is_gtc_order=False, gtc_date='', gtc_end_by=1, state='Pending', created=datetime.datetime(2024, 8, 20, 10, 22, 59, 134000), callbackId=0, threadId='13600')
mit order placed success MITOrderPlaceSuccessResult(threadId='13600', order=PlacedMITOrderResult(account='91829808465', symbol='2888', buysell=0, price=10, qty=1, trigger_price=10, trigger_dir=2, order_flag=0, price_type=2, trade_type=0, is_pre_trade_r


### 5.2.2. 刪除智慧單委託

與一般委託相同，透過 `smartOrder.cancel()` 即可刪除該筆委託

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, User, MITStockOrder, MITOrderSendStartResult, MITOrderSendFailResult, MITOrderSendSuccessResult, MITOrderPlaceFailResult, MITOrderPlaceSuccessResult, MITOrderChangedResult

api.logger.show = False

# 註冊事件
# MIT Order
@api.event.mit_order.send_start
def onMITOrderSendStart(data: MITOrderSendStartResult):
    print(f"MIT order send start: '{data}'")

@api.event.mit_order.send_fail
def onMITOrderSendFail(data: MITOrderSendFailResult):
    print("MIT order send fail", data)

@api.event.mit_order.send_success
def onMITOrderSendSuccess(data: MITOrderSendSuccessResult):
    print(f"MIT order send success: {data}")

@api.event.mit_order.placed_fail
def onMITOrderPlacedFail(data: MITOrderPlaceFailResult):
    print("mit order placed fail", data)  

@api.event.mit_order.placed_success
def onMITOrderPlacedSuccess(data: MITOrderPlaceSuccessResult):
    print("mit order placed success", data)  

@api.event.mit_order.changed
def onMITOrderChanged(data: MITOrderChangedResult):
    print("Smart Order Changed", data)

@api.event.mit_order.cancel_fail
def onMITCancelOrderFailed(data):
    print(f"Cancel MIT Order Failed", data)


api.init()
user: User = api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2888"]
smartOrder: MITStockOrder = stock.smartOrder.mit(api.const.ORDER.ACTION.BUY, 10, 1, 10).send(user.id) 
print(smartOrder) 

smartOrder.cancel()
print(smartOrder)

print("Finished!")
api.keepalive() # 不讓 Notebook 停止

%ZP INFO  2024-08-20 14:29:10.345571 | Main | 38352 |[ eskmo version: 0.0.89 ]
MIT order send start: 'MITOrderSendStartResult(account='91829808465', symbol='2888', buysell=0, price=10, qty=1, trigger_price=10, trigger_dir=2, order_flag=0, price_type=2, trade_type=0, is_pre_trade_risk_controlled=False, is_gtc_order=False, gtc_date='', gtc_end_by=1, callbackId=0)'
MIT order send success: MITOrderSendSuccessResult(account='91829808465', symbol='2888', buysell=0, price=10, qty=1, trigger_price=10, trigger_dir=2, order_flag=0, price_type=2, trade_type=0, is_pre_trade_risk_controlled=False, is_gtc_order=False, gtc_date='', gtc_end_by=1, state='Pending', created=datetime.datetime(2024, 8, 20, 14, 29, 47, 154000), callbackId=0, threadId='30968')
mit order placed fail MITOrderPlaceFailResult(threadId='30968', order={'pid': 38352, 'ClientId': None, 'bstrFullAccount': '91829808465', 'bstrStockNo': '2888', 'bstrPrice': 10, 'sBuySell': 0, 'nQty': 1, 'bstrTriggerPrice': 10, 'bstrDealPrice': '', 'nOr

TypeError: Argument 'smartKeyNo' has incorrect type (expected str, got NoneType)


### 5.2.3. 修改智慧單委託

透過 `smartOrder.modify(price=..., qty=...)` 即可修改該筆智慧單委託

> <br/>
> 建置中
> <br/><br/>